# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR² Dataset) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` fields to ensure consistency and reproducibility.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields by their `@id`s.

In [ ]:
# List all available record sets, fields and columns with their @id references
print('Available Record Sets:')
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"  - RecordSet: {rs['@id']} | name: {rs.get('name','(no name)')}")
        fields = rs.get('fields', [])
        for fld in fields:
            print(f"      Field: {fld['@id']} | name: {fld.get('name','(no name)')} | dataType: {fld.get('dataType','')} ")
            cols = fld.get('columns', [])
            for col in cols:
                print(f"          Column: {col['@id']} | name: {col.get('name','(no name)')}")
else:
    # For older mlcroissant, use dataset.record_sets
    for rs in dataset.record_sets:
        print(f"  - RecordSet: {rs['@id']} | name: {rs.get('name','(no name)')}")
        fields = rs.get('fields', [])
        for fld in fields:
            print(f"      Field: {fld['@id']} | name: {fld.get('name','(no name)')} | dataType: {fld.get('dataType','')} ")
            cols = fld.get('columns', [])
            for col in cols:
                print(f"          Column: {col['@id']} | name: {col.get('name','(no name)')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
We use the record set and field `@id`s as shown in the overview above.

In [ ]:
# List all available record set @id's (utilize overview if you need to pick your own)
record_set_ids = []
if hasattr(metadata, 'record_sets'):
    record_set_ids = [rs['@id'] for rs in metadata.record_sets]
else:
    record_set_ids = [rs['@id'] for rs in dataset.record_sets]

print('RecordSet IDs:', record_set_ids)

# Load all dataframes for all record sets by @id
dataframes = {}
for record_set_id in record_set_ids:
    # mlcroissant expects the @id string for the record_set param
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df

# Display first non-empty DataFrame's columns as an example
for rid, df in dataframes.items():
    if len(df.columns) > 0:
        example_record_set_id = rid
        break

print(f"Fields in record set {example_record_set_id}:\n", dataframes[example_record_set_id].columns.tolist())
display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as record filtering, normalizing numeric fields, and grouping by key attributes.

Replace field `@id` references with those identified in the previous cell.

In [ ]:
# Choose a numeric and a grouping field by @id (edit to match what you found in the overview)
# Example placeholder @id values below (replace with actual @id from data overview)
df = dataframes[example_record_set_id]

# Find numeric columns by dtype
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('Numeric columns identified:', numeric_cols)
if numeric_cols:
    numeric_field_id = numeric_cols[0]
else:
    numeric_field_id = df.columns[0]  # fallback in case none found; edit manually as needed

# Find a categorical/group column (try to use 'Sex', 'Anatomical location', or similar)
candidate_group_fields = [col for col in df.columns if ('sex' in col.lower() or 'location' in col.lower() or 'site' in col.lower())]
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
else:
    group_field_id = df.columns[1] if len(df.columns) > 1 else df.columns[0]

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Filtering: select rows where numeric_field_id > threshold
threshold = 10
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
        print(f"Grouped data by {group_field_id}:")
        display(grouped)
else:
    print(f"Field {numeric_field_id} does not appear numeric. Please check the chosen field.")

## 5. Visualization
Visualize distributions and relationships between fields, using the selected numeric and grouping fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot of numeric field by group
if group_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion
This notebook demonstrates how to load, process, and visualize the FAIR² dataset using the `mlcroissant` package,
consistently referencing each entity by its Croissant `@id`. For more advanced analysis, consider joining fields across record sets or using additional metadata provided within the Croissant schema.